In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import pickle
pd.set_option('display.max_columns', None)

In [3]:
mimic_iv_path = "/mnt/nfs_share/Public_Data/Dataset_MIMICs/physionet.org/files/mimic-iv/2.2/"
mm_dir = "/mnt/data/yihua/master/datasets/mimic-iv"

output_dir = os.path.join(mm_dir, "preprocessing")

In [4]:
restrict_48_hours = True
include_notes = True
include_cxr = False
include_ecg = False
standard_scale = True
include_missing = False

In [5]:
# ireg_vitals_ts_df = pd.read_pickle(os.path.join(output_dir, "ts_vitals_icu.pkl"))
# imputed_vitals = pd.read_pickle(os.path.join(output_dir, "imputed_ts_vitals_icu.pkl"))

ireg_vitals_ts_df = pd.read_pickle(os.path.join(output_dir, "ts_labs_vitals_icu.pkl"))
imputed_vitals = pd.read_pickle(os.path.join(output_dir, "imputed_ts_labs_vitals_icu.pkl"))

ireg_vitals_ts_df = ireg_vitals_ts_df[ireg_vitals_ts_df['timedelta'] >= 0]
imputed_vitals = imputed_vitals[imputed_vitals['timedelta'] >= 0]

if restrict_48_hours:
    ireg_vitals_ts_df = ireg_vitals_ts_df[ireg_vitals_ts_df['timedelta'] <= 48]
    imputed_vitals = imputed_vitals[imputed_vitals['timedelta'] <= 48]

In [6]:
variable_ranges = pd.read_csv('/mnt/nfs_share/Public_Data/Dataset_MIMICs/physionet.org/files/mimic-iv/2.2/variable_ranges.csv')
rename_dict = {
    'Diastolic blood pressure': 'Diastolic BP',
    'Glascow coma scale eye opening': 'GCS - Eye Opening',
    'Glascow coma scale motor response': 'GCS - Motor Response',
    'Glascow coma scale verbal response': 'GCS - Verbal Response',
    'Heart rate': 'Heart Rate',
    'Mean blood pressure': 'Mean BP',
    'Oxygen saturation': 'O2 Saturation',
    'Platelets': 'Platelet Count',
    'Respiratory rate': 'Respiratory Rate',
    'Systolic blood pressure': 'Systolic BP',
    'Blood urea nitrogen': 'Urea Nitrogen',
    'White blood cell count': 'White Blood Cells',
}
variable_ranges['LEVEL2'] = variable_ranges['LEVEL2'].replace(rename_dict)
variable_ranges

,LEVEL2,LEVEL1,OUTLIER LOW,VALID LOW,IMPUTE,VALID HIGH,OUTLIER HIGH
0,Alanine aminotransferase,NaN,0.0,2.00,34.0,10000.00,11000.0
1,Albumin,NaN,0.0,0.60,3.1,6.00,60.0
2,Alkaline phosphate,NaN,0.0,20.00,106.0,3625.00,4000.0
3,Anion Gap,NaN,0.0,5.00,13.0,50.00,55.0
4,Asparate aminotransferase,NaN,0.0,6.00,40.0,20000.00,22000.0
...,...,...,...,...,...,...,...
61,Troponin-I,NaN,0.0,0.01,2.3,49.60,575.0
62,Troponin-T,NaN,0.0,0.01,0.1,20.85,24.0
63,Urine output,NaN,0.0,0.00,80.0,1200.00,2445.0
64,Weight,NaN,0.0,0.00,81.8,250.00,250.0


In [7]:
# Set outliers to NaN
for index, row in variable_ranges.iterrows():
    var_name = row['LEVEL2']
    valid_low = row['VALID LOW']
    valid_high = row['VALID HIGH']
    
    if var_name in ireg_vitals_ts_df.columns and ~np.isnan(valid_low) and ~np.isnan(valid_high):
        ireg_vitals_ts_df[var_name] = ireg_vitals_ts_df[var_name].where(
            ireg_vitals_ts_df[var_name].between(valid_low, valid_high),
            np.nan
        )

# Drop rows that contain NaN for all variable columns
cols = ireg_vitals_ts_df.columns.tolist()
cols = [col for col in cols if col not in ['subject_id', 'hadm_id', 'stay_id', 'timedelta']]
ireg_vitals_ts_df = ireg_vitals_ts_df.dropna(subset=cols, how='all')

In [8]:
ireg_vitals_ts_df.describe()

,subject_id,hadm_id,Anion Gap,Bicarbonate,"Calcium, Total",Chloride,Creatinine,Diastolic BP,GCS - Eye Opening,GCS - Motor Response,GCS - Verbal Response,Glucose,Heart Rate,Hematocrit,Hemoglobin,MCH,MCHC,MCV,Magnesium,Mean BP,Neutrophils,O2 Saturation,Phosphate,Platelet Count,RDW,Red Blood Cells,Respiratory Rate,Sodium,Systolic BP,Urea Nitrogen,Vancomycin,White Blood Cells
count,4.810132e+06,4.810132e+06,207681.000000,209405.000000,185449.000000,219835.000000,211002.000000,2.029295e+06,788519.000000,785090.000000,787143.000000,86222.000000,3.022484e+06,237039.000000,40986.000000,201141.000000,201208.000000,201171.000000,198703.000000,2.028244e+06,33039.000000,2.951162e+06,186354.000000,205578.000000,201060.000000,201176.000000,2.990861e+06,219511.000000,2.029844e+06,210241.000000,9620.000000,201471.000000
mean,1.499283e+07,2.498125e+07,14.555920,23.109791,8.315117,103.854006,1.563522,6.478109e+01,3.345692,5.442808,3.619585,143.883904,8.532980e+01,30.437075,9.708147,30.038363,33.047269,90.983008,2.082141,7.800273e+01,76.894228,9.672596e+01,3.695833,188.470236,15.429358,3.392864,1.928013e+01,138.263618,1.185484e+02,28.321736,17.083482,12.341992
std,2.887041e+06,2.881709e+06,4.423053,4.937612,0.855483,6.934631,1.633295,1.592221e+01,1.025777,1.307203,1.780886,50.274532,1.865423e+01,5.869723,1.993125,2.570723,1.705453,6.947400,0.425613,1.601212e+01,16.147068,3.435402e+00,1.503345,108.802550,2.473442,0.708586,5.771294e+00,5.742689,2.221587e+01,23.358048,8.680309,10.335589
min,1.000003e+07,2.000009e+07,5.000000,2.000000,0.000000,57.000000,0.100000,0.000000e+00,1.000000,1.000000,1.000000,33.000000,0.000000e+00,2.100000,0.000000,0.000000,22.000000,0.000000,0.000000,1.400000e+01,0.000000,0.000000e+00,0.500000,5.000000,0.000000,0.000000,0.000000e+00,80.000000,0.000000e+00,1.000000,0.400000,0.000000
25%,1.248812e+07,2.248825e+07,12.000000,20.000000,7.800000,100.000000,0.700000,5.400000e+01,3.000000,6.000000,1.000000,113.000000,7.200000e+01,26.200000,8.200000,28.800000,32.000000,87.000000,1.800000,6.700000e+01,72.000000,9.500000e+01,2.700000,116.000000,13.700000,2.880000,1.500000e+01,135.000000,1.030000e+02,13.000000,11.100000,7.600000
50%,1.499158e+07,2.497020e+07,14.000000,23.000000,8.300000,104.000000,1.000000,6.300000e+01,4.000000,6.000000,5.000000,134.000000,8.400000e+01,29.800000,9.600000,30.200000,33.100000,91.000000,2.000000,7.600000e+01,80.700000,9.700000e+01,3.400000,171.000000,14.900000,3.330000,1.900000e+01,138.000000,1.160000e+02,20.000000,15.800000,10.700000
75%,1.751312e+07,2.746579e+07,17.000000,26.000000,8.800000,108.000000,1.700000,7.400000e+01,4.000000,6.000000,5.000000,163.000000,9.700000e+01,34.200000,11.000000,31.500000,34.200000,95.000000,2.300000,8.700000e+01,87.000000,9.900000e+01,4.300000,237.000000,16.600000,3.850000,2.200000e+01,141.000000,1.320000e+02,35.000000,21.500000,14.800000
max,1.999999e+07,2.999983e+07,50.000000,50.000000,59.000000,155.000000,36.900000,2.610000e+02,4.000000,6.000000,5.000000,889.000000,2.950000e+02,68.600000,21.000000,58.300000,70.400000,137.000000,19.200000,2.480000e+02,100.000000,1.000000e+02,19.600000,1988.000000,36.000000,7.950000,3.000000e+02,185.000000,3.650000e+02,240.000000,152.800000,572.500000


In [9]:
if include_notes:
    notes_df = pd.read_pickle(os.path.join(output_dir, "icu_notes_text_embeddings.pkl"))
    # notes_df = pd.read_pickle(os.path.join(output_dir, "notes_text.pkl"))
    notes_df = notes_df[notes_df['stay_id'].notnull()]

    notes_df = notes_df[notes_df['icu_time_delta'] >= 0]
    if restrict_48_hours:
        notes_df = notes_df[notes_df['icu_time_delta'] <= 48]

if include_cxr:
    cxr_df = pd.read_pickle(os.path.join(output_dir, "cxr_embeddings_icu.pkl"))
    cxr_df = cxr_df[cxr_df['icu_time_delta'] >= 0]
    if restrict_48_hours:
        cxr_df = cxr_df[cxr_df['icu_time_delta'] <= 48]

if include_ecg:
    ecg_df = pd.read_pickle(os.path.join(output_dir, "ecg_embeddings_icu.pkl"))
    ecg_df = ecg_df[ecg_df['icu_time_delta'] >= 0]
    if restrict_48_hours:
        ecg_df = ecg_df[ecg_df['icu_time_delta'] <= 48]

In [10]:
icustays_df = pd.read_csv(os.path.join(mimic_iv_path, "icu", "icustays.csv"), low_memory=False)
icustays_df['intime'] = pd.to_datetime(icustays_df['intime'])
icustays_df['outtime'] = pd.to_datetime(icustays_df['outtime'])

if restrict_48_hours:
    icustays_df = icustays_df[icustays_df['los'] >= 2]

In [11]:
valid_stay_ids = icustays_df['stay_id'].unique()

ireg_vitals_ts_df = ireg_vitals_ts_df[ireg_vitals_ts_df['stay_id'].isin(valid_stay_ids)]
imputed_vitals = imputed_vitals[imputed_vitals['stay_id'].isin(valid_stay_ids)]

if include_notes:
    notes_df = notes_df[notes_df['stay_id'].isin(valid_stay_ids)]

if include_cxr:
    cxr_df = cxr_df[cxr_df['stay_id'].isin(valid_stay_ids)]

if include_ecg:
    ecg_df = ecg_df[ecg_df['stay_id'].isin(valid_stay_ids)]

In [12]:
admissions_df = pd.read_csv(os.path.join(mimic_iv_path, "hosp", "admissions.csv"))
admissions_df = admissions_df.rename(columns={"hospital_expire_flag": "died"})
admissions_df = admissions_df[["subject_id", "hadm_id", "died"]]

In [13]:

if not include_missing:
    unique_stays = ireg_vitals_ts_df['stay_id'].unique()
    print(f"Number of stays with vitals: {len(unique_stays)}")

    if include_notes:
        unique_stays = np.intersect1d(unique_stays, notes_df['stay_id'].unique())
        print(f"Number of stays with notes: {len(unique_stays)}")

    if include_cxr:
        unique_stays = np.intersect1d(unique_stays, cxr_df['stay_id'].unique())
        print(f"Number of stays with cxr: {len(unique_stays)}")

    if include_ecg:
        unique_stays = np.intersect1d(unique_stays, ecg_df['stay_id'].unique())
        print(f"Number of stays with ecg: {len(unique_stays)}")
else:
    unique_stays = ireg_vitals_ts_df['stay_id'].unique()
    print(f"Number of stays with vitals: {len(unique_stays)}")

    if include_notes:
        # Get stays with either TS or notes
        unique_stays = np.union1d(unique_stays, notes_df['stay_id'].unique())
        print(f"Number of stays with either TS or notes: {len(unique_stays)}")
    
    if include_cxr:
        unique_stays = np.union1d(unique_stays, cxr_df['stay_id'].unique())
        print(f"Number of stays with either TS, notes, cxr: {len(unique_stays)}")
    
    if include_ecg:
        unique_stays = np.union1d(unique_stays, ecg_df['stay_id'].unique())
        print(f"Number of stays with either TS, notes, cxr, ecg: {len(unique_stays)}")

Number of stays with vitals: 35131
Number of stays with notes: 32040


In [14]:
# Create train, val, test splits
np.random.seed(0)
np.random.shuffle(unique_stays)
train_stays = unique_stays[:int(0.7*len(unique_stays))]
val_stays = unique_stays[int(0.7*len(unique_stays)):int(0.85*len(unique_stays))]
test_stays = unique_stays[int(0.85*len(unique_stays)):]

In [16]:
train_ireg_ts_df = ireg_vitals_ts_df[ireg_vitals_ts_df['stay_id'].isin(train_stays)].copy()
train_imputed_df = imputed_vitals[imputed_vitals['stay_id'].isin(train_stays)].copy()

cols = train_ireg_ts_df.columns.tolist()
cols = [col for col in cols if col not in ['subject_id', 'hadm_id', 'stay_id', 'timedelta']]

if standard_scale:
    scalers = {}

    for col in cols:
        scaler = StandardScaler()
        scaler.fit(train_ireg_ts_df[[col]])
        ireg_vitals_ts_df[col] = scaler.transform(ireg_vitals_ts_df[[col]])
        scalers[col] = scaler

        scaler = StandardScaler()
        scaler.fit(train_imputed_df[[col]])
        imputed_vitals[col] = scaler.transform(imputed_vitals[[col]])

base_name = "scalers_ihm"
if restrict_48_hours:
    base_name += "-48"
else:
    base_name += "-all"

if include_notes:
    base_name += "-notes"

if include_cxr:
    base_name += "-cxr"

if include_ecg:
    base_name += "-ecg"

if include_missing:
    base_name += "-missingInd"

f_path = os.path.join(output_dir, f"{base_name}.pkl")
with open(f_path, 'wb') as f:
    pickle.dump(scalers, f)


In [34]:
def get_stay_list(stays):
    stays_list = []

    for curr_stay in tqdm(stays):
        curr_stay_ireg = ireg_vitals_ts_df[ireg_vitals_ts_df['stay_id'] == curr_stay].copy()
        curr_stay_imputed = imputed_vitals[imputed_vitals['stay_id'] == curr_stay].copy()

        if len(curr_stay_ireg) == 0:
            continue

        curr_stay_dict = {}
        curr_stay_dict['name'] = curr_stay_ireg['subject_id'].iloc[0]
        curr_stay_dict['hadm_id'] = curr_stay_ireg['hadm_id'].iloc[0]
        curr_stay_dict['stay_id'] = curr_stay
        curr_stay_dict['ts_tt'] = curr_stay_ireg['timedelta'].values

        curr_stay_ireg.drop(columns=['subject_id', 'hadm_id', 'stay_id', 'timedelta'], inplace=True)
        ireg_ts_mask = curr_stay_ireg.notnull()
        curr_stay_ireg.fillna(0, inplace=True)
        curr_stay_dict['irg_ts'] = curr_stay_ireg.values
        curr_stay_dict['irg_ts_mask'] = ireg_ts_mask.values.astype(int)

        curr_stay_imputed.drop(columns=['subject_id', 'hadm_id', 'stay_id', 'timedelta'], inplace=True)
        curr_stay_dict['reg_ts'] = curr_stay_imputed.values

        if include_notes:
            curr_stay_notes = notes_df[notes_df['stay_id'] == curr_stay].copy()

            if len(curr_stay_notes) == 0:
                curr_stay_dict['text_data'] = []
                curr_stay_dict['text_time_to_end'] = []
                curr_stay_dict['text_embeddings'] = []
                curr_stay_dict['text_missing'] = 1
            else:
                curr_stay_dict['text_data'] = curr_stay_notes['text'].tolist()
                curr_stay_dict['text_time_to_end'] = curr_stay_notes['icu_time_delta'].values
                curr_stay_dict['text_embeddings'] = [emb[0][0] for emb in curr_stay_notes['biobert_embeddings']]
                curr_stay_dict['text_missing'] = 0

        if include_cxr:
            curr_stay_cxr = cxr_df[cxr_df['stay_id'] == curr_stay].copy()
            
            if len(curr_stay_cxr) == 0:
                curr_stay_dict['cxr_feats'] = []
                curr_stay_dict['cxr_time'] = []
                curr_stay_dict['cxr_missing'] = 1
            else:
                curr_stay_dict['cxr_feats'] = curr_stay_cxr['densefeatures'].tolist()
                curr_stay_dict['cxr_time'] = curr_stay_cxr['icu_time_delta'].values
                curr_stay_dict['cxr_missing'] = 0

        if include_ecg:
            curr_stay_ecg = ecg_df[ecg_df['stay_id'] == curr_stay].copy()
            if len(curr_stay_ecg) == 0:
                curr_stay_dict['ecg_feats'] = []
                curr_stay_dict['ecg_time'] = []
                curr_stay_dict['ecg_missing'] = 1
            else:
                curr_stay_dict['ecg_feats'] = curr_stay_ecg['embeddings'].tolist()
                curr_stay_dict['ecg_time'] = curr_stay_ecg['icu_time_delta'].values
                curr_stay_dict['ecg_missing'] = 0

        curr_stay_dict['label'] = admissions_df[admissions_df['hadm_id'] == curr_stay_dict['hadm_id']]['died'].iloc[0]

        stays_list.append(curr_stay_dict)

    return stays_list

train_stays_list = get_stay_list(train_stays)
val_stays_list = get_stay_list(val_stays)
test_stays_list = get_stay_list(test_stays)

100%|██████████| 4806/4806 [37:19<00:00,  2.15it/s]


In [35]:
# Save the data
import pickle

base_name = "ihm"
if restrict_48_hours:
    base_name += "-48"
else:
    base_name += "-all"

if include_notes:
    base_name += "-notes"

if include_cxr:
    base_name += "-cxr"

if include_ecg:
    base_name += "-ecg"

if include_missing:
    base_name += "-missingInd"

f_path = os.path.join(output_dir, f"train_{base_name}_stays.pkl")
with open(f_path, 'wb') as f:
    print(f"Saving train stays to {f_path}")
    pickle.dump(train_stays_list, f)

f_path = os.path.join(output_dir, f"val_{base_name}_stays.pkl")
with open(f_path, 'wb') as f:
    print(f"Saving val stays to {f_path}")
    pickle.dump(val_stays_list, f)

f_path = os.path.join(output_dir, f"test_{base_name}_stays.pkl")
with open(f_path, 'wb') as f:
    print(f"Saving test stays to {f_path}")
    pickle.dump(test_stays_list, f)


Saving train stays to /mnt/data/yihua/master/datasets/mimic-iv_processed/preprocessing/train_ihm-48_stays.pkl
Saving val stays to /mnt/data/yihua/master/datasets/mimic-iv_processed/preprocessing/val_ihm-48_stays.pkl
Saving test stays to /mnt/data/yihua/master/datasets/mimic-iv_processed/preprocessing/test_ihm-48_stays.pkl
